# LOB Data Preview

Read the Bybit BTCUSDT `.data` files in `Data/`, combine them into one lazy table, and preview the raw messages. Only the preview is loaded into memory; the full month stays on disk.

Each row is one message. `bid_updates` and `ask_updates` contain `[price, quantity]` pairs. A `snapshot` contains the book, while a `delta` contains only changes. This notebook does not reconstruct the book or train a model.

In [1]:
from pathlib import Path
from IPython.display import display

try:
    import polars as pl
except ModuleNotFoundError:
    %pip install -q "polars>=2.1,<3"
    import polars as pl

pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(36)
pl.Config.set_fmt_table_cell_list_len(2)
print(f"Polars version: {pl.__version__}")

Polars version: 2.1.0


In [2]:
# Works when the kernel starts in the project root or in LOB/.
candidates = [Path.cwd() / "Data", Path.cwd().parent / "Data"]
DATA_DIR = next((path.resolve() for path in candidates if path.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Open this notebook from the HFT project root or LOB folder.")

files = sorted(DATA_DIR.glob("*_BTCUSDT_ob200.data"))
if not files:
    raise FileNotFoundError(f"No BTCUSDT order-book files found in {DATA_DIR}")

total_bytes = sum(path.stat().st_size for path in files)
print(f"Files: {len(files)}")
print(f"Date range: {files[0].name[:10]} to {files[-1].name[:10]}")
print(f"Raw size: {total_bytes / 1_000_000_000:.2f} GB")

Files: 30
Date range: 2026-09-24 to 2026-10-23
Raw size: 33.75 GB


In [3]:
# Explicit types preserve nested price/quantity lists, including empty deltas.
schema = {
    "topic": pl.String,
    "type": pl.String,
    "ts": pl.Int64,
    "data": pl.Struct({
        "s": pl.String,
        "b": pl.List(pl.List(pl.String)),
        "a": pl.List(pl.List(pl.String)),
        "u": pl.Int64,
        "seq": pl.Int64,
    }),
    "cts": pl.Int64,
}

# lob references every file without loading the whole month into RAM.
lob = (
    pl.scan_ndjson(
        [path.as_posix() for path in files],
        schema=schema,
        include_file_paths="source_file",
        low_memory=True,
    )
    .unnest("data")
    .rename({
        "type": "message_type",
        "s": "symbol",
        "b": "bid_updates",
        "a": "ask_updates",
        "u": "update_id",
        "seq": "sequence",
    })
    .with_columns(
        pl.from_epoch("ts", time_unit="ms")
        .dt.replace_time_zone("UTC").alias("timestamp_utc"),
        pl.col("source_file").str.split("/").list.last(),
    )
    .select(
        "source_file", "timestamp_utc", "symbol", "message_type",
        "bid_updates", "ask_updates", "update_id", "sequence",
        "ts", "cts", "topic",
    )
)

print(f"Combined table: all messages from {len(files)} files (lazy).")

Combined table: all messages from 30 files (lazy).


In [4]:
# Only collect the first 10 rows, not the full 34 GB dataset.
N_PREVIEW_ROWS = 10
preview = lob.head(N_PREVIEW_ROWS).collect()
print(f"Preview: {preview.height} rows x {preview.width} columns")
display(preview)

Preview: 10 rows x 11 columns


source_file,timestamp_utc,symbol,message_type,bid_updates,ask_updates,update_id,sequence,ts,cts,topic
str,"datetime[μs, UTC]",str,str,list[list[str]],list[list[str]],i64,i64,i64,i64,str
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.129 UTC,"""BTCUSDT""","""snapshot""","[[""84355.20"", ""0.949""], … [""84321.50"", ""0.009""]]","[[""84355.30"", ""4.536""], … [""84391.40"", ""0.014""]]",53936686,815049327159,1790208000129,1790208000127,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.229 UTC,"""BTCUSDT""","""delta""","[[""84355.20"", ""0.797""], … [""84321.50"", ""0""]]","[[""84355.30"", ""3.697""], … [""84391.40"", ""0""]]",53936687,815049327497,1790208000229,1790208000227,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.329 UTC,"""BTCUSDT""","""delta""","[[""84355.20"", ""1.444""], … [""84322.00"", ""0""]]","[[""84355.30"", ""3.066""], … [""84391.10"", ""0""]]",53936688,815049327836,1790208000329,1790208000324,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.428 UTC,"""BTCUSDT""","""delta""","[[""84351.00"", ""0.005""], … [""84324.10"", ""0""]]","[[""84359.80"", ""0.015""], … [""84390.40"", ""0.003""]]",53936689,815049327985,1790208000428,1790208000423,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.529 UTC,"""BTCUSDT""","""delta""","[[""84353.00"", ""0""], … [""84324.40"", ""1.659""]]","[[""84358.90"", ""0.017""], … [""84390.80"", ""0""]]",53936690,815049328139,1790208000529,1790208000525,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.629 UTC,"""BTCUSDT""","""delta""","[[""84355.20"", ""1.814""], … [""84322.10"", ""0""]]","[[""84368.20"", ""0.023""], … [""84390.70"", ""1.930""]]",53936691,815049328326,1790208000629,1790208000626,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.729 UTC,"""BTCUSDT""","""delta""","[[""84355.20"", ""1.887""], … [""84322.00"", ""0.119""]]","[[""84355.30"", ""3.070""], … [""84390.70"", ""0""]]",53936692,815049328524,1790208000729,1790208000721,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.828 UTC,"""BTCUSDT""","""delta""","[[""84355.20"", ""1.886""], … [""84322.00"", ""0""]]","[[""84379.90"", ""0.016""], … [""84390.70"", ""1.930""]]",53936693,815049328708,1790208000828,1790208000825,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.928 UTC,"""BTCUSDT""","""delta""","[[""84355.20"", ""1.880""], … [""84321.30"", ""0.002""]]","[[""84355.30"", ""3.068""], … [""84390.40"", ""0.001""]]",53936694,815049328933,1790208000928,1790208000926,"""orderbook.200.BTCUSDT"""


In [5]:
# Inspect the nested values without printing hundreds of price levels.
first = preview.row(0, named=True)
print(f"First message type: {first['message_type']}")
print("First 3 bid pairs [price, quantity]:", first["bid_updates"][:3])
print("First 3 ask pairs [price, quantity]:", first["ask_updates"][:3])

First message type: snapshot
First 3 bid pairs [price, quantity]: [['84355.20', '0.949'], ['84355.10', '0.001'], ['84354.10', '0.001']]
First 3 ask pairs [price, quantity]: [['84355.30', '4.536'], ['84355.40', '0.002'], ['84356.20', '0.002']]
